In [14]:
!pip install dotenv openai pandas openpyxl requests

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [requests]


In [1]:
from agno.agent import Agent
from agno.models.message import Message
from agno.db.sqlite import SqliteDb
from agno.run import RunContext
from agno.models.azure import AzureOpenAI
import os
import pandas as pd
from dotenv import load_dotenv  
from os import getenv
from pathlib import Path  

In [2]:
load_dotenv()
os.environ["AZURE_OPENAI_API_KEY"] = getenv("AZURE_OPENAI_API_KEY")
os.environ["AZURE_ENDPOINT"] = getenv("AZURE_INFERENCE_ENDPOINT")  
os.environ["AZURE_OPENAI_DEPLOYMENT"] = getenv("LLM_MODEL")

In [3]:
model = AzureOpenAI(id=getenv("LLM-MODEL"), 
                    api_version=getenv("LLM_MODEL_VERSION"),
                    azure_endpoint=getenv("AZURE_INFERENCE_ENDPOINT"))
response = model.response(
    messages=[
        Message(role="user", content="Hello")
    ]
)
print(response.content)

Hello! How can I assist you today?


In [4]:
# ─── USER CONFIGURATION ──────────────────────────────────────────────────────
DATA_FOLDER   = Path('../data/input')   # <-- change to your folder

USE_CASE_FILE = DATA_FOLDER / 'use_case_Movie_Recommendation_v2_with_fi.xlsx'
ABT_FILE      = DATA_FOLDER / 'movie_abt_enriched_FINAL.xlsx'
TAXONOMY_FILE = DATA_FOLDER / 'Movie_Recommendation_TaxonomyV2.xlsx'
OSCARS_PDF    = DATA_FOLDER / 'The_98th_Academy_Awards_2026.pdf'

# SQLite file for Agno session memory
MEMORY_DB     = DATA_FOLDER / 'movie_agent_memory.db'
agent_db = SqliteDb(db_file=MEMORY_DB)


AZURE_OPENAI_API_KEY=os.getenv('AZURE_OPENAI_API_KEY')
AZURE_OPENAI_ENDPOINT=os.getenv('AZURE_OPENAI_ENDPOINT')
AZURE_OPENAI_API_VERSION=os.getenv("LLM_MODEL_VERSION") 
TMDB_API_KEY   = os.getenv('TMDB_API_KEY')
SERPER_API_KEY = os.getenv('SERPER_API_KEY')

LLM_MODEL        = os.getenv("LLM_MODEL")
#EMBED_MODEL      = 'text-embedding-3-small'
MAX_ABT_RESULTS  = 5
MAX_TAG_RESULTS  = 10
RAG_TOP_K        = 4
SESSION_ID       = 'movie-bot-session-001'  # change per user

print(f'Data folder: {DATA_FOLDER.resolve()}')
for f in [USE_CASE_FILE, ABT_FILE, TAXONOMY_FILE, OSCARS_PDF]:
    status = '✅' if f.exists() else '❌ NOT FOUND'
    print(f'  {status}  {f.name}')

Data folder: /Users/nsathi/Library/CloudStorage/Dropbox/Applied AI Institute/Courses/GenerativeAI/G-AI-5-Building-Full-Scale-AI-Solutions/AI-Sandbox/Agno-Class-exercises/data/input
  ✅  use_case_Movie_Recommendation_v2_with_fi.xlsx
  ✅  movie_abt_enriched_FINAL.xlsx
  ✅  Movie_Recommendation_TaxonomyV2.xlsx
  ✅  The_98th_Academy_Awards_2026.pdf


In [5]:
df_instructions = pd.read_excel(USE_CASE_FILE, sheet_name='Agent Instructions')
AGENT_SYSTEM_PROMPT = df_instructions[df_instructions['Prompt Type'] == 'agent_prompt'].values[0][1]
print('System prompt loaded:')
print(AGENT_SYSTEM_PROMPT)




System prompt loaded:
You are a friendly and knowledgeable movie expert. Your ONLY role is to help users discover and learn about movies. Do not answer questions unrelated to movies.

UNDERSTAND THE USER (Flipped Interaction)
Before recommending any movie, collect the user's needs through a brief, engaging conversation. Ask ONE leading question at a time across these dimensions (do not ask all at once):
  • Viewing context: Are you watching alone, with family, friends, or a date?
  • Audience: Who is the audience — adults, teens, young children, mixed group?
  • Genre mood: What genre or emotional mood are you in? (e.g., action, comedy, thriller, romance, drama, sci-fi, documentary)
  • Decade/era: Any preference for era or decade? (classic, 1980s–90s, modern, recent)
  • Country/language: Any preference for country of origin or spoken language?
  • Oscar/award interest: Are you interested in critically acclaimed or award-nominated films?
Stop asking when you have enough to make a pers

In [8]:
agent = Agent(
    model=model,
    db=agent_db,
    session_id="My chatbot March 2",
    add_history_to_context=True,
    instructions=AGENT_SYSTEM_PROMPT + "\nCurrent memory is: {chat_history}",
    markdown=True,
    debug_mode=False
)




In [7]:
# Normally we use chat bot like a query engine.  Here is a way to test some queries.
test_queries = [
    "Recommend a romantic comedy for date night.",
    "Who won Best Picture at the 2026 Oscars?",
    "What are the latest Bollywood movies in 2025?",
    "Is Oppenheimer available on Netflix?",
    "Summarize the chat so far"
    ]

for q in test_queries:
    print(f"\nQuery: {q}")
    response = agent.run(q, stream=False)
    print("\nBot:")
    print(response.content)
    print("-" * 80)


Query: Recommend a romantic comedy for date night.

Bot:
Great! Romantic comedy it is. Are you looking for a classic romantic comedy or something more modern and recent? Also, do you prefer any particular setting or cultural backdrop in the movie?
--------------------------------------------------------------------------------

Query: Who won Best Picture at the 2026 Oscars?

Bot:
I specialize in helping with movie recommendations and information. If you want, I can look up the most recent Best Picture winner or help you find a romantic comedy for your date night! Would you like me to do that?
--------------------------------------------------------------------------------

Query: What are the latest Bollywood movies in 2025?

Bot:
I focus on recommending and providing information about movies based on your preferences. Since you're interested in romantic comedies for date night, would you like me to suggest some recent Bollywood romantic comedies or movies that blend romance and come

In [9]:
# This code will demonstrate a real flipped interaction

while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    run = agent.run(user_input, stream=False)
    
    print("\nAgent:")
    print(run.content)


    # ✅ Retrieve latest stored memory
    chat_memory = agent.get_chat_history()


    print("\nSession Memory:")
    for m in chat_memory:
        print("-", m.content)


    print("\n" + "-"*50 + "\n")
#Initial query "Can you recommend a movie for a date night.  
# I love romantic movies and my spouse likes historical fiction"

Your input:  #Initial query ""

Agent:
Great to help you find a movie! To start, are you planning to watch alone, with family, friends, or a date?

Session Memory:
- #Initial query ""
- Great to help you find a movie! To start, are you planning to watch alone, with family, friends, or a date?

--------------------------------------------------

Your input:  Can you recommend a movie for a date night.   # I love romantic movies and my spouse likes historical fiction

Agent:
That’s a wonderful combo for date night! Just to narrow it down, do you prefer a classic or more modern romantic historical fiction movie?

Session Memory:
- #Initial query ""
- Great to help you find a movie! To start, are you planning to watch alone, with family, friends, or a date?
- Can you recommend a movie for a date night.   # I love romantic movies and my spouse likes historical fiction
- That’s a wonderful combo for date night! Just to narrow it down, do you prefer a classic or more modern romantic historica